# DATA209 — Advanced Exploratory Data Analysis
# Practical P5-6 · Unstructured data — image and audio

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 3 · Module 1 · CO1

---

**Objective.** Represent an image as a pixel matrix and an audio file as a waveform, and explore each with the same distributional questions used for tabular data.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.

This session uses the media files `1.png` and `1.wav` instead of the tabular data. If they are not found, the notebook falls back to a built-in sample image and a synthetic tone, so it always runs.

### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

This session works on media files only, so nothing from earlier sessions is needed.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
print("Nothing to rebuild — P5-6 uses image and audio files directly.")
print("Media searched for under DATA_DIR:", os.path.abspath(DATA_DIR))

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P5-6 — Unstructured data — image and audio

### Image pixel distribution

An image is a NumPy array: `(height, width)` for grayscale, `(height, width, 3)` for RGB.
Once it is an array, the same exploratory questions apply — what is the distribution, where are
the extremes, is anything clipped?

In [ ]:
# ---- Load an image -----------------------------------------------------
from PIL import Image

img_path = find("1.png")
if img_path:
    img = np.array(Image.open(img_path).convert("RGB"))
    src = os.path.basename(img_path)
else:
    from skimage import data
    img = data.astronaut()                 # built-in sample, always available
    src = "skimage.data.astronaut()"

print("Source :", src)
print("Shape  :", img.shape, "-> height x width x channels")
print("dtype  :", img.dtype, "| range", img.min(), "to", img.max())
print("Pixels :", f"{img.shape[0] * img.shape[1]:,} per channel")

plt.figure(figsize=(4.2, 4.2))
plt.imshow(img); plt.axis("off"); plt.title(f"Source image — {src}")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Pixel intensity distribution --------------------------------------
gray = np.array(Image.fromarray(img).convert("L"))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

axes[0].imshow(gray, cmap="gray"); axes[0].axis("off")
axes[0].set_title("Grayscale")

axes[1].hist(gray.ravel(), bins=64, color="#3B6E8F")
axes[1].set_title(f"Intensity histogram — mean {gray.mean():.1f}")
axes[1].set_xlabel("pixel value (0-255)")

for ch, colour in zip(range(3), ["#B5432E", "#2E7D4F", "#3B6E8F"]):
    axes[2].hist(img[:, :, ch].ravel(), bins=64, alpha=0.55, color=colour,
                 label=["Red", "Green", "Blue"][ch])
axes[2].legend(); axes[2].set_title("Per-channel distribution")
plt.tight_layout(); plt.show()

stats = pd.DataFrame({
    "channel": ["Red", "Green", "Blue", "Gray"],
    "mean"   : [img[:, :, 0].mean(), img[:, :, 1].mean(), img[:, :, 2].mean(), gray.mean()],
    "std"    : [img[:, :, 0].std(),  img[:, :, 1].std(),  img[:, :, 2].std(),  gray.std()],
    "min"    : [img[:, :, 0].min(),  img[:, :, 1].min(),  img[:, :, 2].min(),  gray.min()],
    "max"    : [img[:, :, 0].max(),  img[:, :, 1].max(),  img[:, :, 2].max(),  gray.max()],
})
print(stats.round(1).to_string(index=False))

clipped_low  = (gray == 0).mean() * 100
clipped_high = (gray == 255).mean() * 100
print(f"\nClipped at 0: {clipped_low:.2f}% of pixels | clipped at 255: {clipped_high:.2f}%")
print("Heavy clipping means detail was lost at capture and cannot be recovered.")
print("A large gap between channel means indicates a colour cast.")

### Simple audio waveform plotting

Audio is amplitude sampled over time: a 1-D array plus a **sample rate**. The standard library
`wave` module reads uncompressed WAV files, so no extra install is needed.

In [ ]:
# ---- Load audio and plot the waveform ----------------------------------
import wave

wav_path = find("1.wav")

if wav_path:
    with wave.open(wav_path, "rb") as w:
        n_channels, sampwidth = w.getnchannels(), w.getsampwidth()
        rate, n_frames        = w.getframerate(), w.getnframes()
        raw = w.readframes(n_frames)
    dtype  = {1: np.uint8, 2: np.int16, 4: np.int32}[sampwidth]
    signal = np.frombuffer(raw, dtype=dtype).astype(np.float32)
    if n_channels > 1:                       # average the channels down to mono
        signal = signal.reshape(-1, n_channels).mean(axis=1)
    signal = signal / (np.abs(signal).max() or 1)     # normalise to -1..1
    src = os.path.basename(wav_path)
else:
    rate, n_channels = 22050, 1
    t = np.linspace(0, 4, rate * 4, endpoint=False)
    signal = (0.6 * np.sin(2 * np.pi * 220 * t) *
              np.exp(-0.4 * t) + 0.03 * np.random.randn(t.size))
    src = "synthetic 220 Hz tone (1.wav not found)"

duration = len(signal) / rate
time     = np.arange(len(signal)) / rate

print("Source      :", src)
print("Sample rate :", f"{rate:,} Hz")
print("Channels    :", n_channels, "(mixed to mono for analysis)")
print("Duration    :", f"{duration:.2f} s")
print("Samples     :", f"{len(signal):,}")

In [ ]:
# ---- Waveform and amplitude distribution -------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

axes[0].plot(time, signal, lw=0.4, color="#3B6E8F")
axes[0].set_title("Waveform"); axes[0].set_xlabel("seconds"); axes[0].set_ylabel("amplitude")

axes[1].hist(signal, bins=80, color="#3B6E8F")
axes[1].set_title("Amplitude distribution"); axes[1].set_xlabel("amplitude")

# short-term energy: loudness over time, in 50 ms frames
frame = int(0.05 * rate)
n_win = len(signal) // frame
energy = np.array([np.sqrt(np.mean(signal[i*frame:(i+1)*frame] ** 2))
                   for i in range(n_win)])
axes[2].plot(np.arange(n_win) * 0.05, energy, color="#B5432E")
axes[2].set_title("Short-term energy (RMS, 50 ms frames)"); axes[2].set_xlabel("seconds")

plt.tight_layout(); plt.show()

silence = (energy < 0.02).mean() * 100
print(f"Peak amplitude   : {np.abs(signal).max():.3f}")
print(f"RMS level        : {np.sqrt(np.mean(signal ** 2)):.3f}")
print(f"Near-silent time : {silence:.1f}% of frames below 0.02 RMS")
print("\nThe amplitude histogram is centred on zero and symmetric — as sound must be.")
print("Energy over time shows structure the raw waveform is too dense to reveal.")

### Deliverable — P5-6

One notebook covering both modalities, stating **shape, dtype and value range explicitly** for
each, plus a short note on what exploratory question each representation makes answerable.